In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error
)

In [9]:
folder = Path("../output_csv")

input_file = (
    folder
    / "xgboost_all_test_predictions.csv"
)

output_file = (
    folder
    / "metrics_summary.csv"
)

print("Input file:", input_file)
print("Output file:", output_file)

Input file: ..\output_csv\xgboost_all_test_predictions.csv
Output file: ..\output_csv\metrics_summary.csv


In [10]:
evaluation_df = pd.read_csv(
    input_file,
    low_memory=False
)

print(
    "Dataset shape:",
    evaluation_df.shape
)

evaluation_df

Dataset shape: (51172, 12)


,model_id,model,training_window_months,train_months,fitting_months,validation_month,test_month,max_depth,learning_rate,n_estimators,actual_price,predicted_price
0,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,"202603, 202604, 202605","202603, 202604",202605,202606,3,0.05,200,2250000.0,1488919.90
1,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,"202603, 202604, 202605","202603, 202604",202605,202606,3,0.05,200,851000.0,1352959.20
2,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,"202603, 202604, 202605","202603, 202604",202605,202606,3,0.05,200,950000.0,1277755.90
3,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,"202603, 202604, 202605","202603, 202604",202605,202606,3,0.05,200,245000.0,616493.90
4,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,"202603, 202604, 202605","202603, 202604",202605,202606,3,0.05,200,1175000.0,903471.10
...,...,...,...,...,...,...,...,...,...,...,...,...
51167,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,"202506, 202507, 202508, 202509, 202510, 202511...","202506, 202507, 202508, 202509, 202510, 202511...",202605,202606,3,0.05,200,40000.0,260401.40
51168,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,"202506, 202507, 202508, 202509, 202510, 202511...","202506, 202507, 202508, 202509, 202510, 202511...",202605,202606,3,0.05,200,865000.0,1842243.40
51169,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,"202506, 202507, 202508, 202509, 202510, 202511...","202506, 202507, 202508, 202509, 202510, 202511...",202605,202606,3,0.05,200,280000.0,954728.30
51170,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,"202506, 202507, 202508, 202509, 202510, 202511...","202506, 202507, 202508, 202509, 202510, 202511...",202605,202606,3,0.05,200,595000.0,1613729.50


In [13]:
# convert to numeric values for evaluation metrics calculations

evaluation_df[
    "actual_price"
] = pd.to_numeric(
    evaluation_df["actual_price"],
    errors="coerce"
)

evaluation_df[
    "predicted_price"
] = pd.to_numeric(
    evaluation_df["predicted_price"],
    errors="coerce"
)

In [14]:
evaluation_df[
    [
        "actual_price",
        "predicted_price"
    ]
].isna().sum()

actual_price       0
predicted_price    0
dtype: int64

In [15]:
# make sure it is above 0 for evaluation metrics calculations

rows_before_cleaning = len(
    evaluation_df
)

evaluation_df = evaluation_df.dropna(
    subset=[
        "actual_price",
        "predicted_price"
    ]
).copy()

evaluation_df = evaluation_df[
    evaluation_df[
        "actual_price"
    ] > 0
].copy()

rows_after_cleaning = len(
    evaluation_df
)

print(
    "Rows before cleaning:",
    rows_before_cleaning
)

print(
    "Rows after cleaning:",
    rows_after_cleaning
)

print(
    "Rows removed:",
    rows_before_cleaning
    - rows_after_cleaning
)

Rows before cleaning: 51172
Rows after cleaning: 51172
Rows removed: 0


## Row-Level Prediction Errors

Absolute error measures the dollar difference between the actual and
predicted price.

Absolute percentage error measures the error as a percentage of the
actual sale price.

In [ ]:
# calculate predicitive error
evaluation_df[
    "prediction_error"
] = (
    evaluation_df[
        "predicted_price"
    ]
    - evaluation_df[
        "actual_price"
    ]
)

In [17]:
# calculate absolute error
evaluation_df[
    "absolute_error"
] = evaluation_df[
    "prediction_error"
].abs()

In [18]:
# calculate  absolute percent error

evaluation_df[
    "absolute_percentage_error"
] = (
    evaluation_df[
        "absolute_error"
    ]
    / evaluation_df[
        "actual_price"
    ].abs()
) * 100

In [ ]:
evaluation_df[
    "absolute_percentage_error"
] = (
    evaluation_df[
        "absolute_error"
    ]
    / evaluation_df[
        "actual_price"
    ].abs()
) * 100

In [19]:
evaluation_df[
    "prediction_direction"
] = np.select(
    [
        evaluation_df[
            "predicted_price"
        ] > evaluation_df[
            "actual_price"
        ],

        evaluation_df[
            "predicted_price"
        ] < evaluation_df[
            "actual_price"
        ]
    ],
    [
        "Overprediction",
        "Underprediction"
    ],
    default="Exact prediction"
)

evaluation_df[
    "prediction_direction"
].value_counts()

prediction_direction
Overprediction     31955
Underprediction    19217
Name: count, dtype: int64

In [21]:
evaluation_df[
    [
        "model_id",
        "actual_price",
        "predicted_price",
        "prediction_error",
        "absolute_error",
        "absolute_percentage_error",
        "prediction_direction"
    ]
].head()

,model_id,actual_price,predicted_price,prediction_error,absolute_error,absolute_percentage_error,prediction_direction
0,XGBoost_3M_depth3_lr0.05_n200,2250000.0,1488919.9,-761080.1,761080.1,33.825782,Underprediction
1,XGBoost_3M_depth3_lr0.05_n200,851000.0,1352959.2,501959.2,501959.2,58.984630,Overprediction
2,XGBoost_3M_depth3_lr0.05_n200,950000.0,1277755.9,327755.9,327755.9,34.500621,Overprediction
3,XGBoost_3M_depth3_lr0.05_n200,245000.0,616493.9,371493.9,371493.9,151.630163,Overprediction
4,XGBoost_3M_depth3_lr0.05_n200,1175000.0,903471.1,-271528.9,271528.9,23.108843,Underprediction


## Evaluation Metrics

R² measures how much variation in actual prices is explained by the
model.

MAE measures the average absolute prediction error in dollars.

RMSE measures prediction error in dollars while giving more weight to
large errors.

MAPE measures the average absolute percentage error.

MdAPE measures the median absolute percentage error. MdAPE is less
sensitive to unusually large errors than MAPE.

In [22]:
def calculate_metrics(group):

    actual = group[
        "actual_price"
    ]

    predicted = group[
        "predicted_price"
    ]

    observations = len(
        group
    )

    if observations >= 2:

        r2 = r2_score(
            actual,
            predicted
        )

    else:

        r2 = np.nan

    mae = mean_absolute_error(
        actual,
        predicted
    )

    rmse = np.sqrt(
        mean_squared_error(
            actual,
            predicted
        )
    )

    mape = (
        mean_absolute_percentage_error(
            actual,
            predicted
        )
        * 100
    )

    mdape = group[
        "absolute_percentage_error"
    ].median()

    return pd.Series(
        {
            "observations": observations,
            "r2": r2,
            "mae": mae,
            "rmse": rmse,
            "mape": mape,
            "mdape": mdape
        }
    )

In [23]:
model_metrics_df = (
    evaluation_df
    .groupby(
        [
            "model_id",
            "model",
            "training_window_months",
            "max_depth",
            "learning_rate",
            "n_estimators",
            "validation_month",
            "test_month"
        ]
    )
    .apply(
        calculate_metrics
    )
    .reset_index()
)

model_metrics_df

C:\Users\luoxu\AppData\Local\Temp\ipykernel_5652\1187817509.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,model_id,model,training_window_months,max_depth,learning_rate,n_estimators,validation_month,test_month,observations,r2,mae,rmse,mape,mdape
0,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,12793.0,0.484318,435269.853919,1.103416e+06,52.067975,25.138328
1,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,12793.0,-0.874438,496778.222464,2.103699e+06,66.284649,25.728637
2,XGBoost_6M_depth3_lr0.05_n200,XGBoost,6,3,0.05,200,202605,202606,12793.0,0.050773,459026.204861,1.497039e+06,56.087100,25.923945
3,XGBoost_9M_depth3_lr0.05_n200,XGBoost,9,3,0.05,200,202605,202606,12793.0,0.392017,442408.019917,1.198102e+06,58.908476,24.763905


In [24]:
# sort by r2

model_metrics_by_r2_df = (
    model_metrics_df
    .sort_values(
        by="r2",
        ascending=False
    )
    .reset_index(
        drop=True
    )
)

model_metrics_by_r2_df

,model_id,model,training_window_months,max_depth,learning_rate,n_estimators,validation_month,test_month,observations,r2,mae,rmse,mape,mdape
0,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,12793.0,0.484318,435269.853919,1.103416e+06,52.067975,25.138328
1,XGBoost_9M_depth3_lr0.05_n200,XGBoost,9,3,0.05,200,202605,202606,12793.0,0.392017,442408.019917,1.198102e+06,58.908476,24.763905
2,XGBoost_6M_depth3_lr0.05_n200,XGBoost,6,3,0.05,200,202605,202606,12793.0,0.050773,459026.204861,1.497039e+06,56.087100,25.923945
3,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,12793.0,-0.874438,496778.222464,2.103699e+06,66.284649,25.728637


In [25]:
#sort by MAPE

model_metrics_by_mape_df = (
    model_metrics_df
    .sort_values(
        by="mape",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)

model_metrics_by_mape_df

,model_id,model,training_window_months,max_depth,learning_rate,n_estimators,validation_month,test_month,observations,r2,mae,rmse,mape,mdape
0,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,12793.0,0.484318,435269.853919,1.103416e+06,52.067975,25.138328
1,XGBoost_6M_depth3_lr0.05_n200,XGBoost,6,3,0.05,200,202605,202606,12793.0,0.050773,459026.204861,1.497039e+06,56.087100,25.923945
2,XGBoost_9M_depth3_lr0.05_n200,XGBoost,9,3,0.05,200,202605,202606,12793.0,0.392017,442408.019917,1.198102e+06,58.908476,24.763905
3,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,12793.0,-0.874438,496778.222464,2.103699e+06,66.284649,25.728637


In [26]:
model_metrics_by_mdape_df = (
    model_metrics_df
    .sort_values(
        by="mdape",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)

model_metrics_by_mdape_df

,model_id,model,training_window_months,max_depth,learning_rate,n_estimators,validation_month,test_month,observations,r2,mae,rmse,mape,mdape
0,XGBoost_9M_depth3_lr0.05_n200,XGBoost,9,3,0.05,200,202605,202606,12793.0,0.392017,442408.019917,1.198102e+06,58.908476,24.763905
1,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,12793.0,0.484318,435269.853919,1.103416e+06,52.067975,25.138328
2,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,12793.0,-0.874438,496778.222464,2.103699e+06,66.284649,25.728637
3,XGBoost_6M_depth3_lr0.05_n200,XGBoost,6,3,0.05,200,202605,202606,12793.0,0.050773,459026.204861,1.497039e+06,56.087100,25.923945


In [27]:
model_comparison_df = model_metrics_df[
    [
        "model_id",
        "training_window_months",
        "max_depth",
        "learning_rate",
        "n_estimators",
        "test_month",
        "observations",
        "r2",
        "mae",
        "rmse",
        "mape",
        "mdape"
    ]
].sort_values(
    by="r2",
    ascending=False
)

model_comparison_df

,model_id,training_window_months,max_depth,learning_rate,n_estimators,test_month,observations,r2,mae,rmse,mape,mdape
0,XGBoost_12M_depth3_lr0.05_n200,12,3,0.05,200,202606,12793.0,0.484318,435269.853919,1.103416e+06,52.067975,25.138328
3,XGBoost_9M_depth3_lr0.05_n200,9,3,0.05,200,202606,12793.0,0.392017,442408.019917,1.198102e+06,58.908476,24.763905
2,XGBoost_6M_depth3_lr0.05_n200,6,3,0.05,200,202606,12793.0,0.050773,459026.204861,1.497039e+06,56.087100,25.923945
1,XGBoost_3M_depth3_lr0.05_n200,3,3,0.05,200,202606,12793.0,-0.874438,496778.222464,2.103699e+06,66.284649,25.728637


## Price-Band Evaluation

The following analysis divides properties into price bands using their
actual sale prices. Model performance is then calculated separately for
each price band.

In [28]:
price_bins = [
    0,
    500_000,
    750_000,
    1_000_000,
    1_500_000,
    2_000_000,
    np.inf
]

price_labels = [
    "Under $500K",
    "$500K-$750K",
    "$750K-$1M",
    "$1M-$1.5M",
    "$1.5M-$2M",
    "$2M+"
]

evaluation_df[
    "price_band"
] = pd.cut(
    evaluation_df[
        "actual_price"
    ],
    bins=price_bins,
    labels=price_labels,
    include_lowest=True,
    right=False
)

In [29]:
evaluation_df[
    "price_band"
].value_counts(
    sort=False
)

price_band
Under $500K     7028
$500K-$750K    10760
$750K-$1M      10440
$1M-$1.5M      10228
$1.5M-$2M       5596
$2M+            7120
Name: count, dtype: int64

In [30]:
first_model_id = evaluation_df[
    "model_id"
].iloc[0]

evaluation_df[
    evaluation_df[
        "model_id"
    ] == first_model_id
][
    "price_band"
].value_counts(
    sort=False
)

price_band
Under $500K    1757
$500K-$750K    2690
$750K-$1M      2610
$1M-$1.5M      2557
$1.5M-$2M      1399
$2M+           1780
Name: count, dtype: int64

In [31]:
price_band_metrics_df = (
    evaluation_df
    .groupby(
        [
            "model_id",
            "model",
            "training_window_months",
            "max_depth",
            "learning_rate",
            "n_estimators",
            "validation_month",
            "test_month",
            "price_band"
        ],
        observed=True
    )
    .apply(
        calculate_metrics
    )
    .reset_index()
)

price_band_metrics_df

C:\Users\luoxu\AppData\Local\Temp\ipykernel_5652\785379801.py:17: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,model_id,model,training_window_months,max_depth,learning_rate,n_estimators,validation_month,test_month,price_band,observations,r2,mae,rmse,mape,mdape
0,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,Under $500K,1757.0,-40.577552,3.124472e+05,5.692202e+05,172.686660,62.483488
1,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,$500K-$750K,2690.0,-45.480166,2.866553e+05,4.905590e+05,46.447154,32.806727
2,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,$750K-$1M,2610.0,-100.770771,2.928384e+05,7.250018e+05,34.416301,20.967102
3,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,$1M-$1.5M,2557.0,-11.352221,2.906790e+05,4.825734e+05,24.157225,17.888896
4,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,$1.5M-$2M,1399.0,-117.602220,4.053593e+05,1.528769e+06,23.887040,16.925511
5,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,$2M+,1780.0,0.447324,1.221159e+06,2.263807e+06,29.627950,28.262179
6,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,Under $500K,1757.0,-458.860101,4.038221e+05,1.893057e+06,237.466812,66.885678
7,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,$500K-$750K,2690.0,-1664.326687,3.924438e+05,2.936345e+06,62.434507,35.208503
8,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,$750K-$1M,2610.0,-271.646008,3.088055e+05,1.186662e+06,35.735652,18.620686
9,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,$1M-$1.5M,2557.0,-255.995469,3.713477e+05,2.201170e+06,31.421061,17.297115


In [33]:
price_band_metrics_df[
    [
        "model_id",
        "training_window_months",
        "price_band",
        "observations",
        "r2",
        "mape",
        "mdape"
    ]
].sort_values(
    by=[
        "training_window_months",
        "price_band"
    ]
)

,model_id,training_window_months,price_band,observations,r2,mape,mdape
6,XGBoost_3M_depth3_lr0.05_n200,3,Under $500K,1757.0,-458.860101,237.466812,66.885678
7,XGBoost_3M_depth3_lr0.05_n200,3,$500K-$750K,2690.0,-1664.326687,62.434507,35.208503
8,XGBoost_3M_depth3_lr0.05_n200,3,$750K-$1M,2610.0,-271.646008,35.735652,18.620686
9,XGBoost_3M_depth3_lr0.05_n200,3,$1M-$1.5M,2557.0,-255.995469,31.421061,17.297115
10,XGBoost_3M_depth3_lr0.05_n200,3,$1.5M-$2M,1399.0,-51.622491,24.139810,18.719673
11,XGBoost_3M_depth3_lr0.05_n200,3,$2M+,1780.0,0.417710,31.132698,29.478112
12,XGBoost_6M_depth3_lr0.05_n200,6,Under $500K,1757.0,-26.331786,181.235265,65.434236
13,XGBoost_6M_depth3_lr0.05_n200,6,$500K-$750K,2690.0,-155.629714,55.115859,37.185681
14,XGBoost_6M_depth3_lr0.05_n200,6,$750K-$1M,2610.0,-1109.583808,39.172733,19.101035
15,XGBoost_6M_depth3_lr0.05_n200,6,$1M-$1.5M,2557.0,-21.752222,24.705236,17.571360


## Create the Metrics Summary

The required `metrics_summary.csv` file will contain both overall model
metrics and price-band metrics.

In [34]:
overall_metrics_df = model_metrics_df.copy()

overall_metrics_df[
    "price_band"
] = "Overall"


metrics_summary_df = pd.concat(
    [
        overall_metrics_df,
        price_band_metrics_df
    ],
    ignore_index=True
)


In [35]:
metrics_summary_df = metrics_summary_df[
    [
        "model_id",
        "model",
        "training_window_months",
        "max_depth",
        "learning_rate",
        "n_estimators",
        "validation_month",
        "test_month",
        "price_band",
        "observations",
        "r2",
        "mae",
        "rmse",
        "mape",
        "mdape"
    ]
]

In [36]:
metrics_summary_df.to_csv(
    folder / "metrics_summary.csv",
    index=False
)

print(
    "Saved:",
    (
        folder
        / "metrics_summary.csv"
    ).resolve()
)

Saved: C:\Users\luoxu\OneDrive - University of Illinois - Urbana\idx-exchange\idx-summer-2026\output_csv\metrics_summary.csv


In [37]:
saved_metrics_summary_df = pd.read_csv(
    folder / "metrics_summary.csv"
)

saved_metrics_summary_df.head()

,model_id,model,training_window_months,max_depth,learning_rate,n_estimators,validation_month,test_month,price_band,observations,r2,mae,rmse,mape,mdape
0,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,Overall,12793.0,0.484318,435269.853919,1.103416e+06,52.067975,25.138328
1,XGBoost_3M_depth3_lr0.05_n200,XGBoost,3,3,0.05,200,202605,202606,Overall,12793.0,-0.874438,496778.222464,2.103699e+06,66.284649,25.728637
2,XGBoost_6M_depth3_lr0.05_n200,XGBoost,6,3,0.05,200,202605,202606,Overall,12793.0,0.050773,459026.204861,1.497039e+06,56.087100,25.923945
3,XGBoost_9M_depth3_lr0.05_n200,XGBoost,9,3,0.05,200,202605,202606,Overall,12793.0,0.392017,442408.019917,1.198102e+06,58.908476,24.763905
4,XGBoost_12M_depth3_lr0.05_n200,XGBoost,12,3,0.05,200,202605,202606,Under $500K,1757.0,-40.577552,312447.153956,5.692202e+05,172.686660,62.483488


## Identify the Best Overall Model

Because Week 8 focuses on percentage-based metrics, the models will be
compared using MdAPE as well as MAPE and R².

In [38]:
overall_comparison_df = (
    metrics_summary_df[
        metrics_summary_df[
            "price_band"
        ] == "Overall"
    ]
    .sort_values(
        by="mdape",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)

overall_comparison_df[
    [
        "model_id",
        "training_window_months",
        "r2",
        "mae",
        "rmse",
        "mape",
        "mdape"
    ]
]

,model_id,training_window_months,r2,mae,rmse,mape,mdape
0,XGBoost_9M_depth3_lr0.05_n200,9,0.392017,442408.019917,1.198102e+06,58.908476,24.763905
1,XGBoost_12M_depth3_lr0.05_n200,12,0.484318,435269.853919,1.103416e+06,52.067975,25.138328
2,XGBoost_3M_depth3_lr0.05_n200,3,-0.874438,496778.222464,2.103699e+06,66.284649,25.728637
3,XGBoost_6M_depth3_lr0.05_n200,6,0.050773,459026.204861,1.497039e+06,56.087100,25.923945


In [39]:
best_model_row = (
    overall_comparison_df
    .iloc[0]
)

best_model_id = best_model_row[
    "model_id"
]

best_training_window = int(
    best_model_row[
        "training_window_months"
    ]
)

print(
    "Best model by MdAPE:",
    best_model_id
)

print(
    "Training window:",
    best_training_window,
    "months"
)

print(
    f"R²: "
    f"{best_model_row['r2']:.4f}"
)

print(
    f"MAPE: "
    f"{best_model_row['mape']:.2f}%"
)

print(
    f"MdAPE: "
    f"{best_model_row['mdape']:.2f}%"
)

Best model by MdAPE: XGBoost_9M_depth3_lr0.05_n200
Training window: 9 months
R²: 0.3920
MAPE: 58.91%
MdAPE: 24.76%


In [40]:
best_r2_model_row = (
    overall_comparison_df
    .sort_values(
        by="r2",
        ascending=False
    )
    .iloc[0]
)

best_mape_model_row = (
    overall_comparison_df
    .sort_values(
        by="mape",
        ascending=True
    )
    .iloc[0]
)

print(
    "Best model by R²:",
    best_r2_model_row[
        "model_id"
    ]
    
)

print(
    "Best model by MAPE:",
    best_mape_model_row[
        "model_id"
    ]
)

print(
    "Best model by MdAPE:",
    best_model_row[
        "model_id"
    ]
)

Best model by R²: XGBoost_12M_depth3_lr0.05_n200
Best model by MAPE: XGBoost_12M_depth3_lr0.05_n200
Best model by MdAPE: XGBoost_9M_depth3_lr0.05_n200


## Price-Band Performance for the Best Model

In [41]:
best_model_price_bands_df = (
    price_band_metrics_df[
        price_band_metrics_df[
            "model_id"
        ] == best_model_id
    ]
    .sort_values(
        by="mdape",
        ascending=True
    )
    .reset_index(
        drop=True
    )
)

best_model_price_bands_df[
    [
        "price_band",
        "observations",
        "r2",
        "mae",
        "rmse",
        "mape",
        "mdape"
    ]
]

,price_band,observations,r2,mae,rmse,mape,mdape
0,$1.5M-$2M,1399.0,-162.364850,3.912562e+05,1.794216e+06,23.128939,15.837788
1,$1M-$1.5M,2557.0,-18.430103,3.211161e+05,6.052411e+05,26.646054,19.036207
2,$750K-$1M,2610.0,-185.824222,3.271296e+05,9.823000e+05,38.334084,20.627011
3,$2M+,1780.0,0.452977,1.188730e+06,2.252200e+06,28.496844,26.418524
4,$500K-$750K,2690.0,-68.788344,2.967266e+05,6.011029e+05,47.940406,31.325052
5,Under $500K,1757.0,-28.511900,2.978499e+05,4.795672e+05,212.514947,58.502223


In [42]:
best_price_band_row = (
    best_model_price_bands_df
    .iloc[0]
)

print(
    "Best-performing price band:",
    best_price_band_row[
        "price_band"
    ]
)

print(
    f"MAPE: "
    f"{best_price_band_row['mape']:.2f}%"
)

print(
    f"MdAPE: "
    f"{best_price_band_row['mdape']:.2f}%"
)

print(
    "Observations:",
    int(
        best_price_band_row[
            "observations"
        ]
    )
)

Best-performing price band: $1.5M-$2M
MAPE: 23.13%
MdAPE: 15.84%
Observations: 1399


In [43]:
worst_price_band_row = (
    best_model_price_bands_df
    .sort_values(
        by="mdape",
        ascending=False
    )
    .iloc[0]
)

print(
    "Worst-performing price band:",
    worst_price_band_row[
        "price_band"
    ]
)

print(
    f"MAPE: "
    f"{worst_price_band_row['mape']:.2f}%"
)

print(
    f"MdAPE: "
    f"{worst_price_band_row['mdape']:.2f}%"
)

print(
    "Observations:",
    int(
        worst_price_band_row[
            "observations"
        ]
    )
)

Worst-performing price band: Under $500K
MAPE: 212.51%
MdAPE: 58.50%
Observations: 1757


In [ ]:
worst_price_band_row = (
    best_model_price_bands_df
    .sort_values(
        by="mdape",
        ascending=False
    )
    .iloc[0]
)

print(
    "Worst-performing price band:",
    worst_price_band_row[
        "price_band"
    ]
)

print(
    f"MAPE: "
    f"{worst_price_band_row['mape']:.2f}%"
)

print(
    f"MdAPE: "
    f"{worst_price_band_row['mdape']:.2f}%"
)

print(
    "Observations:",
    int(
        worst_price_band_row[
            "observations"
        ]
    )
)

In [44]:
mape_mdape_difference = (
    best_model_row[
        "mape"
    ]
    - best_model_row[
        "mdape"
    ]
)

print(
    f"MAPE: "
    f"{best_model_row['mape']:.2f}%"
)

print(
    f"MdAPE: "
    f"{best_model_row['mdape']:.2f}%"
)

print(
    f"Difference: "
    f"{mape_mdape_difference:.2f} "
    "percentage points"
)

MAPE: 58.91%
MdAPE: 24.76%
Difference: 34.14 percentage points


In [45]:
print(
    "Week 8 Evaluation Summary"
)

print(
    "-" * 60
)

print(
    f"The best model by MdAPE was "
    f"{best_model_id}."
)

print(
    f"It used a "
    f"{best_training_window}-month "
    f"training window."
)

print(
    f"Its overall R² was "
    f"{best_model_row['r2']:.4f}."
)

print(
    f"Its overall MAPE was "
    f"{best_model_row['mape']:.2f}%."
)

print(
    f"Its overall MdAPE was "
    f"{best_model_row['mdape']:.2f}%."
)

print(
    f"The best-performing price band was "
    f"{best_price_band_row['price_band']}, "
    f"with an MdAPE of "
    f"{best_price_band_row['mdape']:.2f}%."
)

print(
    f"The worst-performing price band was "
    f"{worst_price_band_row['price_band']}, "
    f"with an MdAPE of "
    f"{worst_price_band_row['mdape']:.2f}%."
)

if (
    best_model_row["mape"]
    > best_model_row["mdape"]
):

    print(
        "MAPE was higher than MdAPE. "
        "This suggests that some unusually "
        "large percentage errors increased "
        "the average error."
    )

else:

    print(
        "MAPE and MdAPE were similar. "
        "This suggests that extreme percentage "
        "errors did not greatly affect the "
        "average error."
    )

Week 8 Evaluation Summary
------------------------------------------------------------
The best model by MdAPE was XGBoost_9M_depth3_lr0.05_n200.
It used a 9-month training window.
Its overall R² was 0.3920.
Its overall MAPE was 58.91%.
Its overall MdAPE was 24.76%.
The best-performing price band was $1.5M-$2M, with an MdAPE of 15.84%.
The worst-performing price band was Under $500K, with an MdAPE of 58.50%.
MAPE was higher than MdAPE. This suggests that some unusually large percentage errors increased the average error.
